# Assignment 6 Neural Architecture Search (NAS)

Change the name of the notebook to: <b><i>StudentName_StudentNumber_ass6.ipynb</i></b>. Submit your solution as a zip file <b><i>StudentName_StudentNumber_ass6.zip</i></b>. Also include your *.ino and *.c *.h file


In assignment 2 we have experimented with manual development of a neural network. However recently more than a thousand papers where published (see [NAS survey paper](./docs/2301.08727.pdf)) about Neural Architecture Search (NAS). This technique aims to automatically find a optimal neural network for a given problem. This way you don't have to be an expert on neural network design to find the optimal hyperparameters and size. Various solutions exist today we take a look at Automated Deep Learning [AutoDL](https://github.com/D-X-Y/AutoDL-Projects), a tool that provides multiple NAS. One of these is [NATS-Bench](https://github.com/D-X-Y/NATS-Bench/blob/main/README.md) (see [paper](docs/2009.00437.pdf)), the successor of NAT-Bench 101 and [Nat-Bench 201](docs/2001.00326.pdf). In this assignment we will analyse the database of already evaluated configurations and query the optimal design for given constraints. We won't run benchmarks ourselfs this requires a lot of computing power and time, this is not feasible for a large group.
It might be interesting for the project to use a NAS to find a solution for your network.

NATS-Bench uses the [CIFAR-10, CIFAR-100](https://www.cs.toronto.edu/~kriz/cifar.html) and ImageNet16-120 dataset. The CIFAR dataset is also image based like Mnist, but contains different classes, e.g. airplane, automobile and bird, CIFAR-100 is an extended version.

Two databases are available the 'archive' (<i>[Google Drive](https://drive.google.com/drive/folders/1zjB6wMANiKwB2A1yil2hQ8H_qyeSe2yt)</i>) and <i>archive-full</i>, both include

- TSS: Topology Search Space $S_t = 6.5k$
- SSS: Size Search Space $S_s = 32.8k$

These contain a file with a benchmark report for the corresponding architecture.
The latter also contains the <b>trained weights</b> for each model. 

We will use 'archive-simple':

- NATS-sss-v1_0-50262-simple.tar
- NATS-tss-v1_0-3ffb9-simple.tar

put them in the .torch/ directory and uncompress with

```console
tar -xvf NATS-sss-v1_0-50262-simple.tar
tar -xvf NATS-tss-v1_0-3ffb9-simple.tar
```

I recommend using a server e.g. Google Collab or Jupyter Utwente for this assignment, some parts are computational intensive and complete much quicker this way.

## Q1

### Q1.1 Explain the concept of a NAS and give the definition

Answer Q1.1:

Neural Architecture Search is a technique that allows for finding a neural network architecture with optimized architecture/hyperparameters for a given need such as performance metrics (faster convergence time) and/or hardware metrics (low power and low memory requirements for embedded systems).

### Q1.2 Explain how to interpret the NATS-Bench figure above (showcases NATS-Bench)

NATS-Bench is a database of already evaluated neural architectures search algorithms that are specialized for particular needs (speed, efficiency etc.). So, one can look at the metrics of the NAS algorithms such as accuracy, performance (convergence time/speed) and efficiency, which are trained and evaluated based on the desired dataset and search space one wants to exploit.


### Q1.3 Setup NATS-Bench using [AutoDL](https://github.com/D-X-Y/AutoDL-Projects/tree/main)
#### - Setup Python environment and needed pip package, see AutoDL docs (tip: use a virtual environment in case of conflicting packages)
#### - Install the 'archive-simple' (2GB no weights) NOT 'archive-full' (680GB 😲 has weights )
#### - Find the <b>train/test accuracy, loss and time</b> for all three datasets (CIFAR-10, CIFAR-100 & ImageNet) for the candidate at index 12 by quering the 'archive-simple'. Explain where to find candidates in the 'archive' with a lower or higher accuracy, how are they ordered?

In [2]:
# ! pip install XAutoDL/

import os
os.environ['TORCH_HOME'] = "./.torch/"

! ls $TORCH_HOME

NATS-sss-v1_0-50262-simple  NATS-tss-v1_0-3ffb9-simple


In [6]:
from nats_bench import create

api = create(None, 'sss', fast_mode=True, verbose=False)

index = 0
info = api.get_more_info(index, 'cifar10')
print(info)

[2026-02-19 23:08:45] Try to use the default NATS-Bench (size) path from fast_mode=True and path=None.
{'train-loss': 0.7731776930999756, 'train-accuracy': 72.334, 'train-per-time': 5.202273607254028, 'train-all-time': 62.42728328704834, 'comment': 'In this dict, train-loss/accuracy/time is the metric on the train+valid sets of CIFAR-10. The test-loss/accuracy/time is the performance of the CIFAR-10 test set after training on the train+valid sets by 12 epochs. The per-time and total-time indicate the per epoch and total time costs, respectively.', 'test-loss': 0.7840251400947571, 'test-accuracy': 72.22, 'test-per-time': 0.5852901935577393, 'test-all-time': 7.023482322692871}


In [ ]:
# Models with highest validation accuracies for each dataset with size search space
cifar10_ilast_more_info = api.get_more_info(api.find_best(dataset='cifar10', metric_on_set='valid')[0], 'cifar10')
pprint('Cifar 10 dataset, model with the highest accuracy:' + str(cifar10_ilast_more_info))

NameError: name 'sss_api' is not defined

### Q1.4 Find the cost <b>FLOPS, PARAMS & latency</b> of the model at index 12 for CIFAR-10. Explain what FLOPS are and why they are important, also explain how to interpret the PARAMS.

In [4]:
# Flops, params and latency info for models for each dataset with size search space
cifar10_i12_cost_info = sss_api.get_cost_info(12, 'cifar10')
pprint('Cifar 10 dataset, model @ index 12:' + str(cifar10_i12_cost_info))
cifar100_i12_cost_info = sss_api.get_cost_info(12, 'cifar100')
pprint('Cifar 100 dataset, model @ index 12:' + str(cifar100_i12_cost_info))
imagenet_i12_cost_info = sss_api.get_cost_info(12, 'ImageNet16-120')
pprint('ImageNet dataset, model @ index 12:' + str(imagenet_i12_cost_info))

## The speed of a computional model, in this case being a neural network model is measured by metrics that indicate the number of operations performed by it per time unit.
## FLOPS (Floating point operations per second is one of these metrics that best showcase the performance of the model (since the weights and calcuations are in fp))
## The higher the params metric, the more complex the model. Reading the paper, one can see that the PARAMS parameter is the size of the model in MB.

("Cifar 10 dataset, model @ index 12:{'flops': 7.991706, 'params': 0.067378, "
 "'latency': 0.014862352974560795, 'T-train@epoch': 5.539922475814819, "
 "'T-train@total': 66.47906970977783, 'T-ori-test@epoch': 0.6709375381469727, "
 "'T-ori-test@total': 8.051250457763672}")
("Cifar 100 dataset, model @ index 12:{'flops': 7.995396, 'params': 0.071068, "
 "'latency': 0.020049911737442017, 'T-train@epoch': 5.594229698181152, "
 "'T-train@total': 67.13075637817383, 'T-x-valid@epoch': 0.5056405067443848, "
 "'T-x-valid@total': 6.067686080932617, 'T-x-test@epoch': 0.3713390827178955, "
 "'T-x-test@total': 4.456068992614746, 'T-ori-test@epoch': 0.6520302295684814, "
 "'T-ori-test@total': 7.824362754821777}")
("ImageNet dataset, model @ index 12:{'flops': 2.002744, 'params': 0.071888, "
 "'latency': 0.019796576764848497, 'T-train@epoch': 11.002268075942993, "
 "'T-train@total': 132.02721691131592, 'T-x-valid@epoch': 0.2767808437347412, "
 "'T-x-valid@total': 3.3213701248168945, 'T-x-test@epoch

### Q1.5 Simulate the training of candidate 12 with 12 epochs for CIFAR-10. Does it match the given test info from before?

In [5]:
cifar10_i12_more_info = sss_api.get_more_info(12, 'cifar10', hp='12')

pprint('Cifar 10 dataset, model @ index 12:' + str(cifar10_i12_more_info) + str(cifar10_i12_cost_info))

sss_api.clear_params(12, hp='12')

validation_accuracy, latency, time_cost, current_total_time_cost = sss_api.simulate_train_eval(12, 'cifar10', hp='12', account_time=False)

print('Validation accuracy: ', validation_accuracy)
print('Latency: ', validation_accuracy)

# There seems to be a mismatch for some reason


("Cifar 10 dataset, model @ index 12:{'train-loss': 0.5464302433204651, "
 "'train-accuracy': 80.77, 'train-per-time': 5.539922475814819, "
 "'train-all-time': 66.47906970977783, 'comment': 'In this dict, "
 'train-loss/accuracy/time is the metric on the train+valid sets of CIFAR-10. '
 'The test-loss/accuracy/time is the performance of the CIFAR-10 test set '
 'after training on the train+valid sets by 12 epochs. The per-time and '
 "total-time indicate the per epoch and total time costs, respectively.', "
 "'test-loss': 0.5867304465293884, 'test-accuracy': 79.9, 'test-per-time': "
 "0.6709375381469727, 'test-all-time': 8.051250457763672}{'flops': 7.991706, "
 "'params': 0.067378, 'latency': 0.014862352974560795, 'T-train@epoch': "
 "5.539922475814819, 'T-train@total': 66.47906970977783, 'T-ori-test@epoch': "
 "0.6709375381469727, 'T-ori-test@total': 8.051250457763672}")
Validation accuracy:  73.21999998046876
Latency:  73.21999998046876


### Q1.6 Show the network of candidate 12 

In [6]:
import xautodl 
from xautodl.models import get_cell_based_tiny_net
config = sss_api.get_net_config(12, 'cifar10')
network = get_cell_based_tiny_net(config)
pprint(network)

ModuleNotFoundError: No module named 'xautodl'

### Q1.7 I have 2 boards the Arduino Nano 33 BLE with a lot of program memory (1MB) and a STM32 Nucleo with 256kB of program memory. Find the best model and its accuracy for each target for the CIFAR-10 dataset. Use the program memory as a constraint for the maximum number of parameters. It is recommended to run this on a server, this is a computational intensive task.

In [ ]:
lim_1mb_index, lim_1mb_acc = sss_api.find_best(dataset='cifar10', metric_on_set='valid', param_max=1)
cifar10_lim_1mb_more_info = sss_api.get_more_info(lim_1mb_index, 'cifar10')
cifar10_lim_1mb_cost_info = sss_api.get_cost_info(lim_1mb_index, 'cifar10')
pprint('Cifar 10 dataset, model with the highest accuracy limited by 1MB:' + str(cifar10_lim_1mb_more_info) + str(cifar10_lim_1mb_cost_info))

lim_256kb_index, lim_1mb_acc = sss_api.find_best(dataset='cifar10', metric_on_set='valid', param_max=0.256)
cifar10_lim_256kb_more_info = sss_api.get_more_info(lim_256kb_index, 'cifar10')
cifar10_lim_256kb_cost_info = sss_api.get_cost_info(lim_256kb_index, 'cifar10')
pprint('Cifar 10 dataset, model with the highest accuracy limited by 256kB:' + str(cifar10_lim_256kb_more_info) + str(cifar10_lim_256kb_cost_info))

('Cifar 10 dataset, model with the highest accuracy limited by '
 "1MB:{'train-loss': 0.2033329369068146, 'train-accuracy': 93.304, "
 "'train-per-time': 10.772470951080322, 'train-all-time': 129.26965141296387, "
 "'comment': 'In this dict, train-loss/accuracy/time is the metric on the "
 'train+valid sets of CIFAR-10. The test-loss/accuracy/time is the performance '
 'of the CIFAR-10 test set after training on the train+valid sets by 12 '
 'epochs. The per-time and total-time indicate the per epoch and total time '
 "costs, respectively.', 'test-loss': 0.32463251345157623, 'test-accuracy': "
 "89.1, 'test-per-time': 0.8606574535369873, 'test-all-time': "
 "10.327889442443848}{'flops': 270.754362, 'params': 0.656506, 'latency': "
 "0.020323960574305786, 'T-train@epoch': 10.772470951080322, 'T-train@total': "
 "129.26965141296387, 'T-ori-test@epoch': 0.8606574535369873, "
 "'T-ori-test@total': 10.327889442443848}")
('Cifar 10 dataset, model with the highest accuracy limited by '
 "256k

### Q1.8 Explain how the authors of NATS-Bench assure a unified performance of each architecture

Answer Q1.8: Assuring a unified performance per architecture implicitly indicates that each model were trained and compared against each other using the same train, test and validation sets. Moreover, the same set of performance metrics (FLOPS, params, latency, time etc.) were used for each architecture to . Also according to the NATS-Bench paper, they used a unified codebase to ensure that the benchmarking is fair as much as possible. They mention a few methods such as the weight-sharing methods being reused, which allows them keep the side-effects of using different NAS algorithms minimal.

In [ ]:
import os
from nats_bench import create

os.environ['TORCH_HOME'] = "./.torch/"

api = create(None, 'sss', fast_mode=True, verbose=False)

[2026-02-20 00:16:24] Try to use the default NATS-Bench (size) path from fast_mode=True and path=None.
Best model index:  12961
Best model test accuracy:  85.655


In [16]:
index, acc = api.find_best(dataset='cifar10', metric_on_set='test', param_max=0.125, hp ='90')

print('Best model index: ', index)
print('Best model test accuracy: ', acc)

Best model index:  9385
Best model test accuracy:  91.1


In [18]:
import pprint
print('Best model index: ', index)
more_info = api.get_more_info(index, 'cifar10', hp='90')
cost_info = api.get_cost_info(index, 'cifar10', hp='90')
pprint.pprint('Cifar 10 dataset, model with the highest accuracy:' + str(more_info) + str(cost_info))

Best model index:  9385
("Cifar 10 dataset, model with the highest accuracy:{'train-loss': "
 "0.09528680694580079, 'train-accuracy': 96.952, 'train-per-time': "
 "6.374523401260376, 'train-all-time': 573.7071061134338, 'comment': 'In this "
 'dict, train-loss/accuracy/time is the metric on the train+valid sets of '
 'CIFAR-10. The test-loss/accuracy/time is the performance of the CIFAR-10 '
 'test set after training on the train+valid sets by 90 epochs. The per-time '
 "and total-time indicate the per epoch and total time costs, respectively.', "
 "'test-loss': 0.30431753048896787, 'test-accuracy': 91.1, 'test-per-time': "
 "0.6561322212219238, 'test-all-time': 59.051899909973145}{'flops': 40.083626, "
 "'params': 0.118402, 'latency': 0.0168751310329048, 'T-train@epoch': "
 "6.374523401260376, 'T-train@total': 573.7071061134338, 'T-ori-test@epoch': "
 "0.6561322212219238, 'T-ori-test@total': 59.051899909973145}")


In [20]:
import xautodl 
from xautodl.models import get_cell_based_tiny_net

config = api.get_net_config(index, 'cifar10')
network = get_cell_based_tiny_net(config)
pprint(network)

ModuleNotFoundError: No module named 'xautodl'

In [ ]:
# Step 2: load weights
import torch

api = create(None, 'sss', fast_mode=False, verbose=True)

model = api.get_net(index, dataset='cifar10', hp='90')

# Step 3: save
torch.save(model.state_dict(), "best_model.pt")